In [2]:
import os
from dotenv import load_dotenv

from typing import TypedDict, Optional, List

from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END


In [4]:
load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


router_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0
)

In [5]:
response = router_llm.invoke("Say hello to SmartPrep AI in one sentence.")
print(response.text)

c:\Users\SHIVAM NAVIK\Desktop\Projects\SmartPrep_AI\genai\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Hello, SmartPrep AI, your ultimate companion for smarter, faster, and more effective preparation!


In [7]:
## adding the state of the graph 


from typing import TypedDict, Optional, List

class SmartPrepState(TypedDict, total=False):
    user_query: str
    intent: str

    resume_text: str
    job_description: str

    uploaded_files: List[str]
    pdf_context: str

    interview_questions: List[str]
    user_answer: str
    evaluation: str

    study_goal: str
    study_plan: str

    web_results: str

    final_response: str
    progress_data: dict


In [8]:

def input_node(state: SmartPrepState):
    print("Input Node Executed")

    user_query = state.get("user_query", "")

    return {
        "user_query": user_query
    }


In [9]:

test_state = {
    "user_query": "Analyze my resume and suggest improvements."
}

result = input_node(test_state)

print(result)


Input Node Executed
{'user_query': 'Analyze my resume and suggest improvements.'}


In [13]:

def router_node(state: SmartPrepState):

    user_query = state["user_query"]

    prompt = f"""
    You are the intent classifier for SmartPrep AI.

    Classify the user's query into exactly one category:

    resume
    job_match
    pdf_qa
    interview
    study_plan
    web_search
    general

    Return ONLY the category name.
    Do not explain your answer.

    User Query: {user_query}
    """

    response = router_llm.invoke(prompt)

    content = response.content

    if isinstance(content, list):
        intent = " ".join(
            item["text"] for item in content
            if isinstance(item, dict) and "text" in item
        )
    else:
        intent = content

    intent = intent.strip().lower()

    valid_intents = [
        "resume",
        "job_match",
        "pdf_qa",
        "interview",
        "study_plan",
        "web_search",
        "general"
    ]

    if intent not in valid_intents:
        intent = "general"

    print("Detected Intent:", intent)

    return {
        "intent": intent
    }


In [14]:
test_state = {
    "user_query": "Analyze my resume and identify my missing skills."
}

result = router_node(test_state)

print(result)

c:\Users\SHIVAM NAVIK\Desktop\Projects\SmartPrep_AI\genai\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Detected Intent: resume
{'intent': 'resume'}


In [15]:

def resume_analysis_node(state: SmartPrepState):

    resume_text = state.get("resume_text", "")

    if not resume_text:
        return {
            "final_response": "Please upload your resume first."
        }

    prompt = f"""
    You are an expert resume analyzer.

    Analyze the following resume and provide:

    1. Candidate Profile
    2. Technical Skills
    3. Education
    4. Work Experience
    5. Projects
    6. Strengths
    7. Missing Skills
    8. Suggestions for Improvement

    Keep the analysis structured and easy to understand.

    Resume:
    {resume_text}
    """

    response = llm.invoke(prompt)

    content = response.content

    if isinstance(content, list):
        final_response = " ".join(
            item["text"] for item in content
            if isinstance(item, dict) and "text" in item
        )
    else:
        final_response = content

    return {
        "final_response": final_response
    }


In [16]:

sample_resume = """
Name: Rahul Sharma

Education:
B.Tech in Computer Science

Skills:
Python, Machine Learning, SQL, TensorFlow, Pandas

Projects:
1. House Price Prediction using Machine Learning
2. Face Recognition Attendance System

Experience:
Machine Learning Intern at ABC Technologies

Worked on data preprocessing and model development.
"""

test_state = {
    "resume_text": sample_resume
}

result = resume_analysis_node(test_state)

print(result["final_response"])


This resume provides a foundational overview of Rahul Sharma's profile. While it highlights key areas, there's significant room for expansion and detail to make it more competitive.

---

### Resume Analysis: Rahul Sharma

#### 1. Candidate Profile
Rahul Sharma appears to be an entry-level Machine Learning enthusiast or recent graduate with a B.Tech in Computer Science. He possesses foundational technical skills in Python, machine learning frameworks, and data manipulation, demonstrated through academic projects and an internship. He is likely seeking roles such as Junior Machine Learning Engineer, Data Scientist (entry-level), or ML Intern.

#### 2. Technical Skills
*   **Programming Languages:** Python
*   **Machine Learning Libraries/Frameworks:** TensorFlow, Pandas, (general "Machine Learning" implies knowledge of Scikit-learn, NumPy, etc., but not explicitly stated)
*   **Databases:** SQL

#### 3. Education
*   **Degree:** B.Tech in Computer Science
    *   *(Missing: University N

In [17]:

def job_match_node(state: SmartPrepState):

    resume_text = state.get("resume_text", "")
    job_description = state.get("job_description", "")

    if not resume_text or not job_description:
        return {
            "final_response": "Please provide both your resume and job description."
        }

    prompt = f"""
    You are an expert technical recruiter.

    Compare the candidate's resume with the given job description.

    Provide the following analysis:

    1. Overall Match Summary
    2. Matching Skills
    3. Missing Skills
    4. Relevant Projects and Experience
    5. Skill Gap Analysis
    6. Suggestions to Improve the Resume
    7. Recommended Topics to Learn

    Do not invent skills or experience that are not present in the resume.

    RESUME:
    {resume_text}

    JOB DESCRIPTION:
    {job_description}
    """

    response = llm.invoke(prompt)

    content = response.content

    if isinstance(content, list):
        final_response = " ".join(
            item["text"] for item in content
            if isinstance(item, dict) and "text" in item
        )
    else:
        final_response = content

    return {
        "final_response": final_response
    }


In [18]:

sample_jd = """
We are hiring a Machine Learning Engineer.

Requirements:
- Strong Python programming
- Knowledge of Machine Learning algorithms
- Experience with TensorFlow or PyTorch
- Knowledge of SQL
- Experience in data preprocessing
- Knowledge of Deep Learning
"""

test_state = {
    "resume_text": sample_resume,
    "job_description": sample_jd
}

result = job_match_node(test_state)

print(result["final_response"])


Here's an analysis of Rahul Sharma's resume against the Machine Learning Engineer job description:

---

### 1. Overall Match Summary

Rahul Sharma presents a strong foundational match for the Machine Learning Engineer role, particularly for an entry-level or junior position. He demonstrates proficiency in core requirements like Python, Machine Learning algorithms, TensorFlow, SQL, and data preprocessing through his skills, projects, and internship experience. The primary area for improvement is explicitly highlighting Deep Learning knowledge and potentially broadening framework experience to include PyTorch.

### 2. Matching Skills

*   **Python programming:** Explicitly listed as a skill.
*   **Knowledge of Machine Learning algorithms:** Explicitly listed as a skill, demonstrated in "House Price Prediction" project and internship.
*   **Experience with TensorFlow:** Explicitly listed as a skill.
*   **Knowledge of SQL:** Explicitly listed as a skill.
*   **Experience in data preproce

In [21]:

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

def process_pdf(pdf_path):

    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_documents(documents)

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name="smartprep_documents"
    )

    return vector_store


In [22]:

def rag_node(state: SmartPrepState):

    question = state["user_query"]
    vector_store = state.get("vector_store")

    if vector_store is None:
        return {
            "final_response": "Please upload and process a PDF first."
        }

    retriever = vector_store.as_retriever(
        search_kwargs={"k": 4}
    )

    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    prompt = f"""
    Answer the user's question using the provided PDF context.

    If the answer is not available in the context,
    clearly say that the uploaded documents do not contain
    sufficient information.

    PDF Context:
    {context}

    User Question:
    {question}
    """

    response = llm.invoke(prompt)

    content = response.content

    if isinstance(content, list):
        final_response = " ".join(
            item["text"] for item in content
            if isinstance(item, dict) and "text" in item
        )
    else:
        final_response = content

    return {
        "pdf_context": context,
        "final_response": final_response
    }


In [24]:
vector_store = process_pdf("new_SHIVAM_Resume.pdf")

c:\Users\SHIVAM NAVIK\Desktop\Projects\SmartPrep_AI\genai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\SHIVAM NAVIK\Desktop\Projects\SmartPrep_AI\genai\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SHIVAM NAVIK\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer 

In [25]:
test_state = {
    "user_query": "Explain the main concepts in this PDF.",
    "vector_store": vector_store
}

result = rag_node(test_state)

print(result["final_response"])


The main concepts in this PDF revolve around **Shivam Navik's professional profile** as an aspiring Artificial Intelligence and Data Science professional. The document highlights:

*   **Education:** Shivam is pursuing a Bachelor of Technology in Artificial Intelligence and Data Science from Jabalpur Engineering College (2022-2026).
*   **Technical Skills:** A broad range of skills including Python, SQL, Machine Learning (Scikit-learn, XGBoost), Deep Learning (TensorFlow, Keras, CNNs), Generative AI (LLMs, RAG, LangChain), Vector Databases (FAISS, ChromaDB), Computer Vision (OpenCV, MediaPipe), Backend & Deployment (FastAPI, Flask, Streamlit), and Databases & Tools (MySQL, Git, GitHub).
*   **Experience:**
    *   **Machine Learning Intern at Prasunet Pvt. Ltd. (Mar-Apr 2025):** Built a handwritten text recognition system with 95%+ accuracy and enhanced model robustness using text correction algorithms.
    *   **Data Analytics and ML Intern at AVTEC Ltd. (Jun-Jul 2023):** Analyzed bre

In [26]:

def interview_node(state: SmartPrepState):

    user_query = state.get("user_query", "")
    resume_text = state.get("resume_text", "")
    user_answer = state.get("user_answer", "")

    if user_answer:

        prompt = f"""
        You are an experienced technical interviewer.

        Evaluate the candidate's answer.

        Provide:
        1. Correctness
        2. Technical Accuracy
        3. Missing Points
        4. Improved Interview Answer
        5. Score out of 10

        Question:
        {user_query}

        Candidate Answer:
        {user_answer}
        """

    else:

        prompt = f"""
        You are an experienced technical interviewer.

        Generate 5 technical interview questions.

        Candidate Resume:
        {resume_text if resume_text else "No resume provided"}

        User Request:
        {user_query}

        Generate relevant questions with varying difficulty:
        Easy, Medium and Hard.
        """

    response = llm.invoke(prompt)

    content = response.content

    if isinstance(content, list):
        final_response = " ".join(
            item["text"] for item in content
            if isinstance(item, dict) and "text" in item
        )
    else:
        final_response = content

    return {
        "final_response": final_response
    }


In [27]:
test_state = {
    "user_query": "Generate Python interview questions for me."
}

result = interview_node(test_state)

print(result["final_response"])

Alright, let's get started. As an experienced technical interviewer, I'm not just looking for correct answers, but also your thought process, problem-solving approach, and ability to articulate complex concepts.

Here are 5 Python interview questions, ranging in difficulty, designed to probe different aspects of your Python knowledge:

---

### Python Technical Interview Questions

**1. Easy: Fundamental Understanding & Data Structures**

*   **Question:** Explain the difference between a `list` and a `tuple` in Python. When would you choose to use a `tuple` over a `list`?
*   **What I'm looking for:**
    *   Understanding of mutability vs. immutability.
    *   Knowledge of common use cases for each.
    *   Ability to articulate basic data structure properties.

**2. Medium: Object-Oriented Programming & Design Patterns**

*   **Question:** What are Python decorators? Provide a simple example of how you might use a decorator to add functionality (e.g., logging, timing) to an existin

In [28]:
test_state = {
    "user_query": "What is the difference between list and tuple in Python?",
    "user_answer": "List is mutable whereas tuple is immutable."
}

result = interview_node(test_state)

print(result["final_response"])

Here's an evaluation of the candidate's answer:

---

### Evaluation of Candidate's Answer

**Candidate Answer:** "List is mutable whereas tuple is immutable."

---

1.  **Correctness:**
    *   **Correct.** The statement is factually accurate. This is the fundamental and most important difference between lists and tuples.

2.  **Technical Accuracy:**
    *   **Accurate.** The terms "mutable" and "immutable" are used correctly and precisely.

3.  **Missing Points:**
    While the candidate identified the core difference, the answer is very brief and lacks depth. A comprehensive answer would include:
    *   **Explanation of Mutability/Immutability:** What does it mean in practice? (e.g., lists can be changed after creation – elements added, removed, modified; tuples cannot be changed after creation).
    *   **Syntax:** How are they defined? (Lists use square brackets `[]`, tuples use parentheses `()`).
    *   **Use Cases/When to Use Which:**
        *   **Lists:** For collections of 

In [29]:

def study_plan_node(state: SmartPrepState):

    study_goal = state.get("study_goal", "")
    user_query = state.get("user_query", "")

    if not study_goal:
        study_goal = user_query

    if not study_goal:
        return {
            "final_response": "Please provide your study goal."
        }

    prompt = f"""
    You are an expert AI learning and career mentor.

    Create a personalized study plan based on the user's goal.

    Include:

    1. Learning objectives
    2. Topics to cover
    3. Daily or weekly schedule
    4. Practical coding exercises
    5. Mini projects
    6. Revision sessions
    7. Final assessment

    Make the plan realistic and beginner-friendly.

    User's Study Goal:
    {study_goal}
    """

    response = llm.invoke(prompt)

    content = response.content

    if isinstance(content, list):
        final_response = " ".join(
            item["text"] for item in content
            if isinstance(item, dict) and "text" in item
        )
    else:
        final_response = content

    return {
        "study_plan": final_response,
        "final_response": final_response
    }


In [30]:
test_state = {
    "user_query": "Create a 30-day Machine Learning study plan.",
    "study_goal": "Learn Machine Learning in 30 days with 3 hours of daily study."
}

result = study_plan_node(test_state)

print(result["study_plan"])

Alright, this is an ambitious but achievable goal if approached with discipline and focus! As your AI learning and career mentor, I'm here to guide you. "Learning Machine Learning in 30 days" means building a strong *foundation* in core concepts and practical implementation, not becoming an expert. We'll focus on supervised and unsupervised learning fundamentals using Python and `scikit-learn`.

Let's craft a personalized, beginner-friendly study plan for you.

---

## Personalized 30-Day Machine Learning Study Plan

**User Goal:** Learn Machine Learning in 30 days with 3 hours of daily study.

**Mentor's Note:** This plan is intensive. Consistency is key. Don't get discouraged if you don't grasp everything immediately; the goal is to build familiarity and practical skills. Focus on understanding *why* things work, not just *how* to run code.

---

### 1. Learning Objectives

By the end of these 30 days, you will be able to:

*   **Understand Core ML Concepts:** Differentiate between s

In [31]:

from ddgs import DDGS

def web_search_node(state: SmartPrepState):

    query = state.get("user_query", "")

    if not query:
        return {
            "final_response": "Please provide a search query."
        }

    try:
        search_results = DDGS().text(
            query,
            max_results=5
        )

        if not search_results:
            return {
                "web_results": "No search results found.",
                "final_response": "Sorry, no relevant information was found."
            }

        formatted_results = []

        for index, result in enumerate(search_results, start=1):

            formatted_results.append(
                f"""
                Result {index}

                Title: {result.get('title', '')}
                URL: {result.get('href', '')}
                Description: {result.get('body', '')}
                """
            )

        web_results = "\n".join(formatted_results)

        prompt = f"""
        You are a helpful career and study research assistant.

        Summarize the following web search results
        according to the user's query.

        Use only the information provided in the results.
        Do not invent facts.

        User Query:
        {query}

        Search Results:
        {web_results}

        Include relevant source URLs in your answer.
        """

        response = llm.invoke(prompt)

        content = response.content

        if isinstance(content, list):
            final_response = " ".join(
                item["text"] for item in content
                if isinstance(item, dict) and "text" in item
            )
        else:
            final_response = content

        return {
            "web_results": web_results,
            "final_response": final_response
        }

    except Exception as e:
        return {
            "final_response": f"Web search failed: {str(e)}"
        }


In [33]:
test_state = {
    "user_query": "Latest Python interview questions for freshers"
}

result = web_search_node(test_state)

print(result["final_response"])

For freshers preparing for Python interviews, several resources offer updated questions and answers, often for the year 2026.

Key information for freshers includes:
*   **Guides for 2026** Many guides are available for Python interviews in 2026, specifically catering to freshers and experienced professionals alike, with some offering over 70 or 120 top questions and answers ([Result 2](https://www.upgrad.com/blog/python-interview-questions-answers/), [Result 4](https://www.interviewbit.com/python-interview-questions/)). One resource provides "top 50 Python interview questions for freshers" ([Result 5](https://www.placementpreparation.io/blog/python-interview-questions-for-freshers/)).
*   **Common Topics** Interview questions for freshers often cover Python basics, syntax, Object-Oriented Programming (OOP), data structures, functions, decorators, generators, modules, and general coding basics ([Result 3](https://pythonguides.com/python-interview-questions-and-answers/), [Result 4](htt

In [34]:

def chat_node(state: SmartPrepState):

    user_query = state.get("user_query", "")

    prompt = f"""
    You are SmartPrep AI, an intelligent study and career assistant.

    Help the user with:
    - Programming
    - Data Structures and Algorithms
    - Artificial Intelligence
    - Machine Learning
    - Deep Learning
    - Generative AI
    - Career guidance
    - Technical interview preparation

    Give clear, accurate and beginner-friendly answers.

    User Query:
    {user_query}
    """

    response = llm.invoke(prompt)

    content = response.content

    if isinstance(content, list):
        final_response = " ".join(
            item["text"] for item in content
            if isinstance(item, dict) and "text" in item
        )
    else:
        final_response = content

    return {
        "final_response": final_response
    }


In [35]:
test_state = {
    "user_query": "Explain the difference between Python list and tuple."
}

result = chat_node(test_state)

print(result["final_response"])

That's a great question! Understanding the difference between Python lists and tuples is fundamental for any Python programmer. Both are used to store collections of items, but they have a crucial distinction.

Here's a breakdown:

---

### Python List vs. Tuple: The Key Differences

The most important difference lies in their **mutability**.

#### 1. Mutability (The BIG Difference!)

*   **Lists are Mutable:**
    *   This means you **can change** a list after it has been created. You can add, remove, or modify elements within the list.
    *   Think of it like a shopping cart: you can add items, remove items, or change the quantity of an item.

    **Example (List):**
    ```python
    my_list = [1, 2, 3]
    print("Original list:", my_list) # Output: Original list: [1, 2, 3]

    # Add an element
    my_list.append(4)
    print("After appending:", my_list) # Output: After appending: [1, 2, 3, 4]

    # Modify an element
    my_list[0] = 10
    print("After modifying:", my_list) # Ou

In [36]:

def response_node(state: SmartPrepState):

    final_response = state.get("final_response", "")

    if not final_response:
        final_response = "Sorry, I could not generate a response."

    return {
        "final_response": final_response.strip()
    }


In [37]:
test_state = {
    "final_response": "Your resume matches the Machine Learning Engineer role well."
}

result = response_node(test_state)

print(result["final_response"])

Your resume matches the Machine Learning Engineer role well.


In [38]:

import json
import os
from datetime import datetime

def save_progress_node(state: SmartPrepState):

    progress_file = "smartprep_progress.json"

    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as file:
            progress_data = json.load(file)
    else:
        progress_data = []

    record = {
        "timestamp": datetime.now().isoformat(),
        "user_query": state.get("user_query", ""),
        "intent": state.get("intent", ""),
        "final_response": state.get("final_response", ""),
        "study_plan": state.get("study_plan", ""),
        "evaluation": state.get("evaluation", "")
    }

    progress_data.append(record)

    with open(progress_file, "w", encoding="utf-8") as file:
        json.dump(progress_data, file, indent=4, ensure_ascii=False)

    print("Progress saved successfully!")

    return {
        "progress_data": record
    }


In [39]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(SmartPrepState)

# Register all nodes
builder.add_node("input_node", input_node)
builder.add_node("router_node", router_node)

builder.add_node("resume_analysis_node", resume_analysis_node)
builder.add_node("job_match_node", job_match_node)
builder.add_node("rag_node", rag_node)
builder.add_node("interview_node", interview_node)
builder.add_node("study_plan_node", study_plan_node)
builder.add_node("web_search_node", web_search_node)
builder.add_node("chat_node", chat_node)

builder.add_node("response_node", response_node)

# Starting flow
builder.add_edge(START, "input_node")
builder.add_edge("input_node", "router_node")

# Conditional routing
def route_request(state: SmartPrepState):
    return state["intent"]

builder.add_conditional_edges(
    "router_node",
    route_request,
    {
        "resume": "resume_analysis_node",
        "job_match": "job_match_node",
        "pdf_qa": "rag_node",
        "interview": "interview_node",
        "study_plan": "study_plan_node",
        "web_search": "web_search_node",
        "general": "chat_node"
    }
)

# Connect every workflow to response node
workflow_nodes = [
    "resume_analysis_node",
    "job_match_node",
    "rag_node",
    "interview_node",
    "study_plan_node",
    "web_search_node",
    "chat_node"
]

for node in workflow_nodes:
    builder.add_edge(node, "response_node")

# End the graph
builder.add_edge("response_node", END)

# Compile
graph = builder.compile()

print("SmartPrep AI Graph compiled successfully!")


SmartPrep AI Graph compiled successfully!


In [40]:
test_state = {
    "user_query": "Explain the difference between machine learning and deep learning."
}

result = graph.invoke(test_state)

print("Intent:", result.get("intent"))
print("\nSmartPrep AI Response:\n")
print(result.get("final_response"))

Input Node Executed


c:\Users\SHIVAM NAVIK\Desktop\Projects\SmartPrep_AI\genai\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Detected Intent: general
Intent: general

SmartPrep AI Response:

That's a great question to start with! The terms "Machine Learning" and "Deep Learning" are often used interchangeably, but they have distinct meanings and a specific relationship.

Let's break them down in a beginner-friendly way.

---

### What is Machine Learning (ML)?

Imagine you want to teach a computer to do something, but you don't want to write explicit rules for every single scenario. That's where Machine Learning comes in!

*   **Definition:** Machine Learning is a field of Artificial Intelligence (AI) that enables systems to **learn from data** without being explicitly programmed. Instead of giving the computer step-by-step instructions, you give it a lot of examples (data), and it figures out the patterns and rules on its own.
*   **How it works:**
    1.  **Data:** You provide a dataset with examples.
    2.  **Feature Engineering (often manual):** You (the human) identify and extract relevant characteristi